**CHARACTER LEVEL LANGUAGE MODEL**

TRAIN A PYTORCH RNN TO GENERATE SHAKESPEAREN TEXT CHARACTER BY CHARACTER.

In [2]:
#Importing necessary packages

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [3]:
# Load Corpus (Subsampled for Speed)
with open("/content/tiny shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# FOR FAST RESULTS: Use only the first 50,000 characters
text = text[:50000]

In [4]:
# Character Vocabulary

chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

In [5]:
# Memory-Efficient Custom Dataset

class ShakespeareDataset(Dataset):
    def __init__(self, text, char_to_idx, seq_length=50):
        self.encoded = [char_to_idx[c] for c in text]
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded) - self.seq_length

    def __getitem__(self, idx):
        x = self.encoded[idx : idx + self.seq_length]
        y = self.encoded[idx + 1 : idx + self.seq_length + 1]
        return torch.tensor(x), torch.tensor(y)

sequence_length = 50
dataset = ShakespeareDataset(text, char_to_idx, sequence_length)


In [6]:
# Increased batch_size to 256 to speed up processing loop drastically
batch_size = 256
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
# Character RNN

class CharRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        output, hidden = self.rnn(x, hidden)
        logits = self.fc(output)
        return logits, hidden


In [8]:
# Execution Settings
embedding_dim = 64
hidden_size = 128

# Automatically use hardware acceleration if you turned on GPU in Colab
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running training on: {device.type.upper()}")

model = CharRNN(vocab_size, embedding_dim, hidden_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

Running training on: CUDA


In [9]:
# Training Loop

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    num_batches = 0

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()

        output, _ = model(xb)
        loss = criterion(output.reshape(-1, vocab_size), yb.reshape(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss / num_batches:.4f}")


Epoch 1/5 | Loss: 1.9429
Epoch 2/5 | Loss: 1.4761
Epoch 3/5 | Loss: 1.3252
Epoch 4/5 | Loss: 1.2422
Epoch 5/5 | Loss: 1.1968


In [12]:
print(model.eval())

CharRNN(
  (embedding): Embedding(59, 64)
  (rnn): RNN(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=59, bias=True)
)


In [14]:
# Text Generation
seed = "W"
generated = seed
hidden = None
current = torch.tensor([[char_to_idx[seed]]]).to(device)

with torch.no_grad():
    for _ in range(300):
        output, hidden = model(current, hidden)
        probs = torch.softmax(output[:, -1, :], dim=-1)
        next_char_idx = torch.multinomial(probs, 1).item()

        generated += idx_to_char[next_char_idx]
        current = torch.tensor([[next_char_idx]]).to(device)

print("Generated Text:")
print(generated)


Generated Text:
Whomeeting,
By that you'll have striculords is not sewald Lettle good speak bodven senats hit iswity bliese ever him opeed to by the common courtest,
And seells,
And consent they have paulwixn oar toing a
ffrilsity!

First Citizen:
I can marcald:
And tale
it brighed?

MELERINIUS:
Go, Marcius do Rome 
